# Hospital Council OpenEnv: Colab RL Training Notebook

This notebook is a Colab-friendly end-to-end training pipeline for the `hospital_council_env` OpenEnv submission.

It does four things:

1. Clones the repo and installs the environment plus training dependencies.
2. Starts the local OpenEnv FastAPI server in synthetic bootstrap mode.
3. Runs baseline-vs-random evaluation before training.
4. Launches the TRL GRPO training script and plots the resulting trainer logs.

The notebook is designed so judges can run it on Colab without access to private MIMIC files.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/BaibhavSureka/meta.git"
REPO_DIR = Path("/content/meta")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
print("Repo files:", sorted(item.name for item in Path.cwd().iterdir())[:12])

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        "./hospital_council_env[train]",
        "matplotlib",
        "requests",
    ],
    check=True,
)
print("Dependencies installed.")

In [ ]:
import json
import time

import requests
import torch

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Notebook will still run on CPU, but training will be much slower.")

## Start the local OpenEnv server

The public Space runs in synthetic bootstrap mode when private MIMIC data is unavailable. We use the same mode here so the notebook is fully reproducible.

In [ ]:
repo_root = Path.cwd()
env_dir = repo_root / "hospital_council_env"
output_dir = repo_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
server_log = output_dir / "colab_server.log"
base_url = "http://127.0.0.1:8000"

server_env = os.environ.copy()
server_env["PORT"] = "8000"
server_env["PYTHONUNBUFFERED"] = "1"
server_env.setdefault("TOKENIZERS_PARALLELISM", "false")

server_handle = open(server_log, "w", encoding="utf-8")
server_process = subprocess.Popen(
    [sys.executable, "-m", "hospital_council_env.server.app"],
    cwd=str(env_dir),
    env=server_env,
    stdout=server_handle,
    stderr=subprocess.STDOUT,
)

for attempt in range(60):
    try:
        response = requests.get(f"{base_url}/status", timeout=2)
        if response.ok:
            print("Server status:", response.json())
            break
    except Exception:
        time.sleep(2)
else:
    server_handle.flush()
    print(server_log.read_text(encoding="utf-8")[-4000:])
    raise RuntimeError("OpenEnv server failed to start.")

In [ ]:
from hospital_council_env.server.hospital_council_env_environment import HospitalCouncilEnvironment
from hospital_council_env.training.evaluate_policy import choose_policy, evaluate_policy

evidence_dir = repo_root / "docs" / "evidence" / "colab_training"
evidence_dir.mkdir(parents=True, exist_ok=True)

baseline_env = HospitalCouncilEnvironment(data_root="missing-mimic-root", sample_size=100)
baseline_metrics = evaluate_policy(baseline_env, choose_policy("baseline", 7), episodes=8)

random_env = HospitalCouncilEnvironment(data_root="missing-mimic-root", sample_size=100)
random_metrics = evaluate_policy(random_env, choose_policy("random", 7), episodes=8)

(evidence_dir / "pretrain_baseline_metrics.json").write_text(json.dumps(baseline_metrics, indent=2), encoding="utf-8")
(evidence_dir / "pretrain_random_metrics.json").write_text(json.dumps(random_metrics, indent=2), encoding="utf-8")

summary = {
    "baseline_avg_reward": round(baseline_metrics["avg_reward"], 4),
    "random_avg_reward": round(random_metrics["avg_reward"], 4),
    "baseline_success_rate": round(baseline_metrics["success_rate"], 4),
    "random_success_rate": round(random_metrics["success_rate"], 4),
    "baseline_task_graph_loss": round(baseline_metrics["avg_task_graph_loss"], 4),
    "random_task_graph_loss": round(random_metrics["avg_task_graph_loss"], 4),
}
print(json.dumps(summary, indent=2))

In [ ]:
import matplotlib.pyplot as plt

metric_names = ["avg_reward", "success_rate", "avg_task_graph_loss"]
baseline_values = [baseline_metrics[name] for name in metric_names]
random_values = [random_metrics[name] for name in metric_names]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric_name, baseline_value, random_value in zip(axes, metric_names, baseline_values, random_values):
    ax.bar(["baseline", "random"], [baseline_value, random_value], color=["#1f77b4", "#ff7f0e"])
    ax.set_title(metric_name)
    ax.grid(axis="y", alpha=0.2)

fig.suptitle("Pre-training policy comparison")
fig.tight_layout()
comparison_plot = evidence_dir / "pretrain_policy_comparison.png"
fig.savefig(comparison_plot, dpi=160, bbox_inches="tight")
plt.show()
print("Saved plot to:", comparison_plot)

## Launch GRPO training

The default model below matches the repository's existing training script. If your Colab runtime is very limited, reduce `DATASET_SIZE` first before changing the model.

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"
DATASET_SIZE = 32
TRAIN_OUTPUT_DIR = repo_root / "outputs" / "colab_grpo_hospital_council"

train_env = os.environ.copy()
train_env["HOSPITAL_COUNCIL_ENV_URL"] = base_url
train_env.setdefault("TOKENIZERS_PARALLELISM", "false")

train_command = [
    sys.executable,
    "-m",
    "hospital_council_env.training.hf_trl_grpo_minimal",
    "--model",
    MODEL_ID,
    "--output-dir",
    str(TRAIN_OUTPUT_DIR),
    "--dataset-size",
    str(DATASET_SIZE),
]

print("Running:", " ".join(train_command))
subprocess.run(train_command, check=True, env=train_env)

In [ ]:
trainer_state_candidates = list(TRAIN_OUTPUT_DIR.rglob("trainer_state.json"))
if not trainer_state_candidates:
    raise FileNotFoundError(f"No trainer_state.json found under {TRAIN_OUTPUT_DIR}")

trainer_state_path = trainer_state_candidates[0]
trainer_state = json.loads(trainer_state_path.read_text(encoding="utf-8"))
log_history = trainer_state.get("log_history", [])
print("Trainer state:", trainer_state_path)
print("Logged steps:", len(log_history))

steps = []
loss_values = []
reward_values = []
for row in log_history:
    step = row.get("step")
    if step is None:
        continue
    if "loss" in row:
        steps.append(step)
        loss_values.append(row["loss"])
    if "reward" in row:
        reward_values.append((step, row["reward"]))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
if steps and loss_values:
    axes[0].plot(steps, loss_values, marker="o")
    axes[0].set_title("Training loss")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("loss")
    axes[0].grid(alpha=0.2)
else:
    axes[0].text(0.5, 0.5, "No loss values logged", ha="center", va="center")
    axes[0].set_axis_off()

if reward_values:
    reward_steps = [item[0] for item in reward_values]
    reward_scores = [item[1] for item in reward_values]
    axes[1].plot(reward_steps, reward_scores, marker="o")
    axes[1].set_title("Reward trace")
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("reward")
    axes[1].grid(alpha=0.2)
else:
    axes[1].text(0.5, 0.5, "No reward values logged", ha="center", va="center")
    axes[1].set_axis_off()

fig.tight_layout()
training_plot = evidence_dir / "grpo_training_curves.png"
fig.savefig(training_plot, dpi=160, bbox_inches="tight")
plt.show()
print("Saved training curves to:", training_plot)
print("Recent log rows:")
for row in log_history[-5:]:
    print(row)

In [ ]:
server_process.terminate()
try:
    server_process.wait(timeout=10)
except subprocess.TimeoutExpired:
    server_process.kill()

server_handle.close()
print("Server stopped.")
print("Notebook artifacts saved under:", evidence_dir)